# Reproducing the submission

This notebook rebuilds my Kaggle submission from scratch: it downloads the trained weights from
the GitHub release, runs the model over the 180 test images, turns the predictions into filament
instances and writes the submission CSV. At the end it compares the result with the CSV I
actually submitted (`submissions/submission_8_0.4508_high0.995_low0.4_min_area400.csv`).

**Before running:** install `requirements.txt` (PyTorch from the CUDA 12.6 index, see the README)
and put the competition data under `Data/`. Only the test images are needed for this notebook.

Inference takes about 3 minutes on an RTX 2050 (4 GB). Training is optional and described in
the last section.

## 1. Setup

In [1]:
import hashlib
import os
import sys
import urllib.request

import numpy as np
import torch

sys.path.insert(0, os.path.abspath("../src"))
import filament as F

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("test images found:", len(F.list_test_images()))
print("post-processing:", F.SUBMISSION_CFG)

torch 2.13.0+cu126 | CUDA available: True
test images found: 180
post-processing: {'low': 0.4, 'high': 0.995, 'min_area': 400}


## 2. Download the trained weights

The weights (280 MB) are attached to release `v1.0` of this repository. The file is checked
against its SHA-256, so the model used here is exactly the one behind the submission.

In [2]:
WEIGHTS_URL = ("https://github.com/ii56/Solar-Filament-Segmentation-Challenge-2026/"
               "releases/download/v1.0/unet_resnet34_clahe_bce_tversky_cosine_best.pt")
WEIGHTS_SHA256 = "179939b0d1939386b1fbc955d15674af02cc2acce483355cc928fa3eb99f0865"
CKPT = os.path.join(F.CHECKPOINT_DIR, "unet_resnet34_clahe_bce_tversky_cosine_best.pt")

os.makedirs(F.CHECKPOINT_DIR, exist_ok=True)
if not os.path.exists(CKPT):
    print("downloading weights ...")
    urllib.request.urlretrieve(WEIGHTS_URL, CKPT)
else:
    print("weights already present:", CKPT)

digest = F.checkpoint_sha256(CKPT)
assert digest == WEIGHTS_SHA256, f"checksum mismatch: {digest}"
print("SHA-256 OK")

weights already present: C:\Users\Ibaan Ibrahim\Documents\Coding\Hackathon\Solar Filament Segmentation Challenge 2026\checkpoints\unet_resnet34_clahe_bce_tversky_cosine_best.pt


SHA-256 OK


## 3. Predict the test set

For each image: CLAHE, 25 overlapping 512x512 tiles, the network, averaging of the overlapping
tile probabilities, then hysteresis thresholding (regions of p > 0.4 kept only if their peak
reaches 0.995 and they cover at least 400 px). The instance maps go to a fresh folder under
`cache/`, so nothing from earlier runs is reused.

In [3]:
model, ck = F.load_model(CKPT)
RESULTS = F.predict_test_instances(model, "reproduced", checkpoint_path=CKPT)

loaded unet_resnet34_clahe_bce_tversky_cosine_best.pt: epoch 8, best val IoU 0.4508


  1/180 images


  31/180 images


  61/180 images


  91/180 images


  121/180 images


  151/180 images


  180/180 images


180 test images in C:\Users\Ibaan Ibrahim\Documents\Coding\Hackathon\Solar Filament Segmentation Challenge 2026\cache\test_reproduced_high0.995_low0.4_min_area400 (180 computed, 0 cached)


## 4. Write and verify the CSV

In [4]:
OUT = os.path.join(F.PROJECT_ROOT, "submissions", "reproduced_submission.csv")
info = F.write_submission_csv(RESULTS, OUT)
check = F.verify_submission_csv(OUT, RESULTS)

print(f"wrote {OUT}")
print(f"{info['rows']} rows for {info['images'] - len(info['images_without_rows'])} of "
      f"{info['images']} images")
print("problems:", check["problems"] or "none")
assert not check["problems"] and check["mismatched_masks"] == 0

wrote C:\Users\Ibaan Ibrahim\Documents\Coding\Hackathon\Solar Filament Segmentation Challenge 2026\submissions\reproduced_submission.csv
1442 rows for 177 of 180 images
problems: none


## 5. Compare with the submitted file

Rows are compared mask by mask (decoded, not as strings). GPU convolutions are not always
bit-for-bit deterministic, so a handful of pixels right at a threshold could in principle differ;
the pixel agreement below shows how close the two files are.

In [5]:
from pycocotools import mask as maskUtils

SUBMITTED = os.path.join(F.PROJECT_ROOT, "submissions",
                         "submission_8_0.4508_high0.995_low0.4_min_area400.csv")


def masks_by_image(path):
    out = {}
    for fid, counts in F.read_submission_csv(path):
        stem = fid.rpartition("_")[0]
        m = maskUtils.decode({"size": [2048, 2048], "counts": counts.encode("ascii")}) > 0
        out.setdefault(stem, []).append(m)
    return out


a, b = masks_by_image(SUBMITTED), masks_by_image(OUT)
same_rows = dict(F.read_submission_csv(SUBMITTED)) == dict(F.read_submission_csv(OUT))
inter = union = 0
for stem in set(a) | set(b):
    ma = np.any(a.get(stem, [np.zeros((2048, 2048), bool)]), axis=0)
    mb = np.any(b.get(stem, [np.zeros((2048, 2048), bool)]), axis=0)
    inter += int((ma & mb).sum())
    union += int((ma | mb).sum())

print(f"rows: submitted {sum(map(len, a.values()))}, reproduced {sum(map(len, b.values()))}")
print(f"identical file content: {same_rows}")
print(f"pixel IoU between the two submissions: {inter / union:.6f}")

rows: submitted 1442, reproduced 1442
identical file content: True
pixel IoU between the two submissions: 1.000000


## 6. Training (optional, about 9 GPU-hours)

The weights above come from two training stages, both run with `F.train`:

1. Adam at a constant learning rate of 1e-4, batch size 4, mixed precision, from ImageNet
   weights. I kept the checkpoint with the best validation tile IoU, which was epoch 12.
2. From those weights, 10 more epochs with cosine decay from 1e-4 to 1e-6. The submitted model
   is epoch 8 of this stage.

Training was not seeded, so a rerun gives similar but not bit-identical weights. Set
`RUN_TRAINING = True` to run it; on a 4 GB GPU, close everything else that uses the GPU first.

In [6]:
RUN_TRAINING = False

if RUN_TRAINING:
    ctx = F.setup()                                   # builds the tile index on first run
    # stage 1: constant learning rate, best checkpoint by validation IoU
    F.train(ctx, epochs=12, lr=1e-4)
    # stage 2: cosine decay from the stage-1 best weights, under a new tag
    model = F.build_model()
    stage1_best = os.path.join(F.CHECKPOINT_DIR, F.TAG + "_best.pt")
    F.train(ctx, model, epochs=10, lr=1e-4, resume=stage1_best, new_run=True,
            tag=F.TAG + "_cosine", scheduler_fn=F.cosine_schedule(10))
    # the stage-2 best checkpoint is checkpoints/unet_resnet34_clahe_bce_tversky_cosine_best.pt